In [ ]:
Klastry obliczeniowe i rozwiązania chmurowe 2026/5_1
Zadania do samodzielnego wykonania (Projekt końcowy)
Temat: PySpark w Databricks - analiza FASTQ.
pd5071

In [1]:
# 
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("FASTQ_Analysis")
    .master("local[*]")
    .getOrCreate()
)

spark



In [2]:
# Wczytanie pliku
file_path = "/home/clusters/work/SRR16356246_1_1.fastq"

lines_df = spark.read.text(file_path)

# Numeracja linii
window = Window.orderBy(monotonically_increasing_id())

lines_with_id = lines_df.withColumn(
    "line_number",
    row_number().over(window) - 1
)

# Typ linii FASTQ
lines_typed = lines_with_id.withColumn(
    "line_type",
    when(col("line_number") % 4 == 0, "header")
    .when(col("line_number") % 4 == 1, "sequence")
    .when(col("line_number") % 4 == 2, "separator")
    .otherwise("quality")
)

# Identyfikator rekordu
lines_with_record = lines_typed.withColumn(
    "record_id",
    (col("line_number") / 4).cast("int")
)

# Pivot
fastq_wide = lines_with_record.groupBy("record_id").pivot("line_type").agg(
    first("value")
)


In [3]:
# Zadania 1 - Walidacja integralności danych
validated = (
    fastq_wide
    .withColumn(
        "declared_length",
        regexp_extract(col("header"), r"length=(\d+)", 1).cast("int")
    )
    .withColumn(
        "actual_length",
        length(col("sequence"))
    )
)

invalid_records = validated.filter(
    col("declared_length") != col("actual_length")
)

print("Liczba niespójnych rekordów:", invalid_records.count())
print("Liczba partycji:", lines_df.rdd.getNumPartitions())


Liczba niespójnych rekordów: 0
Liczba partycji: 1


In [ ]:
Analiza wykonania w UI Zadania 1 - Walidacja integralności danych

Cel zadania: sprawdzenie spójności danych. Zweryfikujemy, czy deklarowana długość
sekwencji w nagłówku pliku FASTQ odpowiada jej rzeczywistej długości. Jest to fundamentalny
krok w kontroli jakości danych przed rozpoczęciem jakiejkolwiek analizy.

● Ile Jobs zostało utworzonych? Czy widzimy zależność między liczbą wywołań akcji a
liczbą zadań?
    
Liczba Jobs: 2 (1/1 i 2-gi skipped)
Teoretycznie kod zawiera 1 akcję (.count()), więc powinien powstać 1 job 
– zależność 1 akcja = 1 job.
W praktyce Spark utworzył dwa Joby, ponieważ włączony mechanizm 
Adaptive Query Execution (AQE) może rozdzielać wykonanie planu na dodatkowe Joby 
związane z operacjami Exchange (shuffle) i optymalizacją planu wykonania. 
Reguła „1 akcja = 1 Job” nie zawsze obowiązuje przy włączonym AQE.


● Które operacje były transformacjami, a które akcjami? Wypiszmy je, aby utrwalić tę
kluczową różnicę.

Transformacje:
- .withColumn("declared_length", regexp_extract(...)) - dodaje nową kolumnę
- .withColumn("actual_length", length(...)) - dodaje nową kolumnę
- .filter(col("declared_length") != col("actual_length")) - filtruje wiersze

Akcja:
- .count() - wykonuje obliczenia i zwraca wynik

● Sprawdźmy DAG Visualization dla tego Joba. Ile etapów (RDD/DataFrame) tworzy plan
wykonania zapytania?

Operacje walidacji (regexp_extract, length, filter) są transformacjami wąskimi. 
Jednak plan wykonania obejmuje cały lineage prowadzący do utworzenia DataFrame fastq_wide, 
dlatego w DAG występują również wcześniejsze operacje Window, Exchange (SinglePartition) oraz SortAggregate. 
Operacje te wymagają przetasowania danych (shuffle), co powoduje podział wykonania na 3 Stages. 
Jeden ze Stages został oznaczony jako Skipped, ponieważ Spark ponownie wykorzystał 
wcześniej zmaterializowany wynik dzięki mechanizmowi Adaptive Query Execution (AQE).

● Ile tasków wykonał jedyny Stage w tym Jobie? Porównajmy tę liczbę z domyślną liczbą
partycji DataFrame. Wyciągnijmy wniosek o relacji między partycjami a taskami.

Liczba Stages: 3 (w tym 1 skipped z 0 tasków, 2 aktywne po 1 tasku).
Liczba Partycji: 1

Teoretycznie relacja jest 1:1 – liczba tasków stage powinna być równa liczbie partycji.
Każdy aktywny Stage wykonał 1 task. Odpowiada to 1 partycji danych. 
Relacja liczby tasków do liczby partycji wynosi więc 1:1. 
Jedna partycja wynika z operacji Exchange SinglePartition zastosowanej przy 
row_number() bez partitionBy oraz z niewielkiego rozmiaru pliku. 
Liczba partycji określa stopień równoległości przetwarzania. 
W tym przypadku dane były przetwarzane w jednej partycji, co ogranicza możliwość równoległego wykonania obliczeń.

Podsumowanie:
Rozbieżność względem prostego modelu wykonania wynika z dwóch czynników:
- włączonego mechanizmu Adaptive Query Execution (AQE), który może zmieniać relację między akcjami a Jobami,
- faktu, że Spark analizuje cały lineage DataFrame. Operacje Window, Exchange i SortAggregate z etapu parsowania FASTQ
wpływają na kształt DAG oraz liczbę Stages.

In [4]:
# Zadanie 2 – Wstępna analiza jakości

low_quality = fastq_wide.filter(
    col("quality").contains("#")
)

print("Liczba odczytów o niskiej jakości:", low_quality.count())

Liczba odczytów o niskiej jakości: 0


In [ ]:
Zadanie 2 Wstępna analiza jakości

Cel zadania: zidentyfikować odczyty, które zawierają bazę o bardzo niskiej jakości. W
kodowaniu Phred+33, znak # odpowiada jakości równej 2, co jest praktycznie zerowym
zaufaniem do poprawności odczytania nukleotydu.

● Porównajmy czas wykonania tego zadania z Zadaniem 1. Które było szybsze i
dlaczego? (podpowiedź: porównajmy złożoność operacji regexp_extract i length z
prostym contains).

Zadanie 2 było szybsze (ok. 120 ms wobec ok. 250 ms w Zadaniu 1).
Operacja contains('#') wykonuje jedynie proste przeszukanie tekstu, 
natomiast w Zadaniu 1 wykonywane były bardziej kosztowne operacje: 
regexp_extract(), length(), konwersja typu oraz porównanie wartości. 
Dodatkowo w Zadaniu 2 przetwarzana była tylko kolumna quality, a nie header i sequence.

● Czy Spark wykorzystał cache? Przyjrzyjmy się liście stage. Czy któryś z nich ma przy
sobie zieloną etykietę "skipped"? Jeśli tak, oznacza to, że Spark nie musiał ponownie
czytać danych z dysku, ponieważ skorzystał z wyniku zapisanego w pamięci przez
.cache() w poprzednim kroku.

Nie. Cache nie został wykorzystany, 
ponieważ DataFrame nie został wcześniej zapisany w pamięci za pomocą cache() lub persist().
Widoczne w Spark UI etapy Skipped wynikają z mechanizmu AQE (Adaptive Query Execution), 
który ponownie wykorzystuje wyniki wcześniejszych operacji shuffle, a nie z użycia pamięci podręcznej (cache).

● Jaki był Locality Level tasków? W szczegółach Stage'a znajdźmy informację o "Locality Level". 
Czy dominowało PROCESS_LOCAL (najlepsza opcja, dane w pamięci tego samego procesora)? Jeśli tak, dlaczego?

DDominował PROCESS_LOCAL, ponieważ dane i executor znajdowały się na tym samym węźle (local[*]). 
Dzięki temu zadania były wykonywane lokalnie, bez przesyłania danych przez sieć.

● Ile rekordów (Records) przetworzył każdy Task? Czy liczba ta jest zgodna z Twoimi
oczekiwaniami dotyczącymi podziału danych na partycje?

Każdy task przetworzył 400 linii wejściowych, co po zgrupowaniu odpowiada 100 rekordom FASTQ. 
Dane były przetwarzane w jednej partycji, dlatego wykonany został jeden task. 
Potwierdza to zależność 1 partycja = 1 task.

In [5]:

# Zadanie 3 - Kompleksowa analiza i optymalizacja

from pyspark.sql.functions import col, length

# Kompleksowa analiza
quality_stats = (
    fastq_wide
    .withColumn("has_low_quality", col("quality").contains("#"))
    .withColumn("seq_length", length(col("sequence")))
    .groupBy("has_low_quality", "seq_length")
    .count()
    .orderBy("has_low_quality", "seq_length")
    .cache()
)

# Materializacja cache
quality_stats.count()

# Kolejne akcje wykorzystujące cache
quality_stats.select("seq_length").distinct().count()

quality_stats.show(truncate=False)


+---------------+----------+-----+
|has_low_quality|seq_length|count|
+---------------+----------+-----+
|false          |151       |100  |
+---------------+----------+-----+



In [ ]:
Zadanie - 3 Kompleksowa analiza i optymalizacja\

Cel zadania: połączenie wszystkich poznanych operacji w jedno, złożone zapytanie analityczne,
które zbada zależność między długością sekwencji a występowaniem w niej niskiej jakości.
Następnie zoptymalizujemy jego wykonanie poprzez cache'owanie wyniku pośredniego.


Kompleksowa analiza w Spark UI
● Ile Jobs zostało utworzonych? Wypiszmy każdą akcję z kodu i przypisz jej numer Joba z UI.

Utworzono 5 Jobów (Job 6–10).

quality_stats.count() → Job 6 i 7 (materializacja cache)
quality_stats.select("seq_length").distinct().count() → Job 8 i 9
quality_stats.show() → Job 10

Dodatkowe Joby wynikają z działania AQE, które może rozdzielać wykonanie na etapy związane z operacjami shuffle.
    
● Narysujmy schemat Stages dla pierwszego Joba (tego z groupBy):

Job 6–7 (pierwsza akcja)

Stage 9:
Scan text
-> Project
-> Exchange (Shuffle Write: 13,2 KiB)

Stage 11:
-> Sort
-> Window
-> Project
-> SortAggregate
-> HashAggregate
-> Sort
-> zapis do cache

    ○ Stage 1: Jakie operacje się tu odbywały? Czy widzimy w metrykach Shuffle Write? 
        (to etap, na którym taski przygotowują dane do agregacji).

    Stage 9 (Shuffle Write)

    Scan text → Project → Exchange
    Input: 35,5 KiB
    Shuffle Write: 13,2 KiB
        W tym etapie Spark odczytał dane z pliku, wykonał wstępne transformacje i przygotował dane do agregacji, 
        zapisując je w obszarze shuffle.

    ○ Stage 2: Jakie operacje? Czy widzimy Shuffle Read? (to etap agregacji, na
        którym taski czytają dane przygotowane przez Stage 1).

    Stage 11 (Shuffle Read)

    Sort → Window → Project → SortAggregate → HashAggregate → Sort → zapis do cache
    Shuffle Read: 13,2 KiB
        W tym etapie Spark odczytał dane przygotowane podczas shuffle, 
        wykonał kolejne transformacje (m.in. Window i SortAggregate), następnie agregację groupBy().count(), 
        posortował wynik i zapisał go w pamięci podręcznej (cache) do wykorzystania przez kolejne akcje.

● Które Jobs wykorzystały cache? Zidentyfikujmy Joby, których etapy zostały pominięte ("skipped"). 
Dlaczego tak się stało?

Cache został utworzony podczas wykonania Joba 7, natomiast Joby 8–10 korzystają już z wyniku zapisanego w pamięci. 
Potwierdza to zakładka Storage (Cached Partitions: 1, Fraction Cached: 100%, Size in Memory: 448 B). 
Dodatkowo w zakładce Stages Joby 8–10 mają Input = 448 B, co oznacza, że Spark odczytuje niewielki, 
zagregowany wynik z pamięci zamiast ponownie czytać plik FASTQ (35,5 KiB).

Etapy oznaczone jako skipped oznaczają, że Spark nie wykonywał ponownie wcześniej obliczonych fragmentów planu. 
W Jobie 7 pominięty stage wynika z działania Adaptive Query Execution (AQE), 
które ponownie wykorzystuje wcześniej wykonany etap shuffle. 
Natomiast w Jobach 8–10 pomijany jest pełny pipeline odczytu i przetwarzania danych, 
ponieważ kolejne akcje korzystają z wyniku zapisnego w cache.

● Porównajmy Input Size między różnymi Jobami. Dlaczego Jobs korzystające z cache
mają bardzo mały lub zerowy rozmiar wejściowy?

Job 6: 35,5 KiB (odczyt pliku FASTQ)
Job 7: Shuffle Read = 13,2 KiB(odczyt danych przygotowanych w etapie shuffle)
Joby 8–10: Input Size ≈ 448 B (odczyt danych z cache)
                                                
Po wykonaniu pierwszej akcji wynik zapytania został zapisany w pamięci (cache). 
Kolejne Joby nie odczytują już pliku FASTQ ani nie wykonują ponownie pełnego pipeline'u przetwarzania. 
Zamiast tego korzystają z niewielkiego, zagregowanego wyniku zapisnego w cache, 
dlatego ich Input Size jest wielokrotnie mniejszy (448 B zamiast 35,5 KiB).

● Efektywność cache: porównajmy czas wykonania pierwszego Joba (który musiał
obliczyć wszystko od zera) z czasami kolejnych Jobów (które czytały z cache). Jaki jest
zysk?
| Job    |  Czas | Opis                                                                                       |
| ------ | ----- | ------------------------------------------------------------------------------------------ |
| Job 6  | 0,1 s | Odczyt danych z pliku FASTQ i przygotowanie danych do shuffle.                             |
| Job 7  | 0,2 s | Dokończenie obliczeń (agregacja, sortowanie, zapis wyniku do cache i wykonanie `count()`). |
| Job 8  | 83 ms | Odczyt wyniku z cache i wykonanie `distinct().count()`.                                    |
| Job 10 | 66 ms | Odczyt wyniku z cache i wykonanie `show()`.                                                |

Pierwsza akcja wymagała wykonania Jobów 6 i 7, ponieważ Spark (AQE) podzielił jej realizację na dwa etapy. 
Kolejne akcje (Joby 8–10) korzystały już z danych zapisanych w cache,
dlatego wykonywały się około 2–3 razy szybciej i nie wymagały ponownego odczytu oraz przetwarzania pliku FASTQ.

● Locality Level: Czy wszystkie taski w Jobach korzystających z cache miały
PROCESS_LOCAL? Jeśli nie, spróbujmy określić, dlaczego niektóre taski musiały
czytać dane z pamięci na innym węźle (NODE_LOCAL)

Można to sformułować krócej i bardziej w stylu poprzednich odpowiedzi:

Wszystkie taski w Jobach korzystających z cache miały Locality Level = PROCESS_LOCAL.

Wynika to z uruchomienia aplikacji w trybie local[*], gdzie dane i executor działają w tym samym procesie. 
Dodatkowo wynik został zapisany w pamięci jako jedna partycja (Cached Partitions: 1, Size in Memory: 448 B), 
dzięki czemu Spark odczytywał dane bezpośrednio z pamięci, bez konieczności transferu przez sieć.

W środowisku wielowęzłowym mógłby pojawić się NODE_LOCAL, gdy dane znajdowałyby się na tym samym węźle, 
lecz były przechowywane w pamięci innego executora niż ten wykonujący task. 
W analizowanym przypadku taka sytuacja nie wystąpiła.